# SQL practice 8: dates, times and time series

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql/sql-practice-08-dates-and-time-series.ipynb)

14 exercises on parsing, date arithmetic, time zones, spines, rolling windows, sessions and cohorts. Each exercise names the [reference entries](sql-reference-08-dates-and-time-series.ipynb) it practises, shows the exact expected output, and hides a reference solution you can reveal when you are done or stuck.

*Last reviewed: November 2026, against Apache Spark 4.0 (Spark SQL).*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local Spark session, the shared tables as temporary views: `customers`, `orders`, `order_items`, `products`, `events` and `employees` (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry, two pairs of employees tie on salary, Dana out-earns her manager and Jo has no department), and the helpers from [SQL reference 8](sql-reference-08-dates-and-time-series.ipynb#setup-and-sample-data). Exercises that need other data create it in their input cell.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
# Shared sample tables as temporary views. Each statement is ordinary Spark SQL.
TABLES = {
    "customers": """CREATE OR REPLACE TEMP VIEW customers AS SELECT * FROM VALUES
  (1, 'Acme Corp',      'North', DATE'2024-01-03'),
  (2, 'Bright Retail',  'North', DATE'2024-02-11'),
  (3, 'Cyan Traders',   'South', DATE'2024-03-15'),
  (4, 'Delta Grocers',  NULL,    DATE'2024-05-02'),   -- region never arrived from the CRM
  (5, 'Echo Supplies',  'West',  DATE'2024-06-20'),
  (6, 'Foxtrot Foods',  'West',  DATE'2024-08-01'),
  (7, 'Gamma Sports',   'East',  DATE'2024-09-14'),   -- 7 and 8 have never ordered
  (8, 'Hotel Homeware', 'East',  DATE'2024-11-05')
AS customers(customer_id, name, region, signup_date)""",
    "orders": """CREATE OR REPLACE TEMP VIEW orders AS SELECT order_id, customer_id, status, order_ts, CAST(amount AS DECIMAL(12,2)) AS amount FROM VALUES
  (1,  1, 'paid',     TIMESTAMP'2026-01-05 09:15:00', 120.00),
  (2,  1, 'shipped',  TIMESTAMP'2026-01-05 10:05:00', 84.50),
  (3,  1, 'returned', TIMESTAMP'2026-01-05 11:40:00', 30.00),
  (4,  2, 'paid',     TIMESTAMP'2026-01-06 15:20:00', 260.00),
  (5,  2, 'placed',   TIMESTAMP'2026-01-07 08:05:00', 55.25),
  (6,  3, 'shipped',  TIMESTAMP'2026-01-07 17:45:00', 149.00),
  (7,  3, 'paid',     TIMESTAMP'2026-01-09 12:10:00', 310.00),
  (8,  4, 'returned', TIMESTAMP'2026-01-10 19:30:00', 89.90),
  (9,  4, 'shipped',  TIMESTAMP'2026-01-12 08:55:00', 1180.00),
  (10, 5, 'paid',     TIMESTAMP'2026-01-12 13:00:00', 74.00),
  (11, 6, 'placed',   TIMESTAMP'2026-01-13 10:25:00', 210.00),
  (12, 6, 'shipped',  TIMESTAMP'2026-01-14 00:30:00', 96.40)   -- just after midnight UTC
AS orders(order_id, customer_id, status, order_ts, amount)""",
    "order_items": """CREATE OR REPLACE TEMP VIEW order_items AS SELECT order_id, sku, qty, CAST(unit_price AS DECIMAL(8,2)) AS unit_price FROM VALUES
  (1, 'SKU-4001', 2, 12.50), (1, 'SKU-4007', 1, 95.00), (2, 'SKU-4002', 1, 34.50), (2, 'SKU-4009', 2, 25.00),
  (3, 'SKU-4002', 1, 30.00), (4, 'SKU-4003', 1, 149.00), (4, 'SKU-4004', 2, 55.50), (5, 'SKU-4005', 1, 55.25),
  (6, 'SKU-4003', 1, 149.00), (7, 'SKU-4006', 1, 189.00), (7, 'SKU-4008', 1, 121.00), (8, 'SKU-4010', 1, 89.90),
  (9, 'SKU-4011', 20, 59.00), (10, 'SKU-4012', 2, 37.00), (11, 'SKU-4013', 1, 210.00),
  (12, 'SKU-4014', 1, 41.40), (12, 'SKU-4005', 1, 55.00)
AS order_items(order_id, sku, qty, unit_price)""",
    "products": """CREATE OR REPLACE TEMP VIEW products AS SELECT sku, category, CAST(list_price AS DECIMAL(8,2)) AS list_price FROM VALUES
  ('SKU-4001', 'Kitchen', 12.50), ('SKU-4002', 'Kitchen', 34.50), ('SKU-4003', 'Garden', 149.00),
  ('SKU-4004', 'Garden', 55.50), ('SKU-4005', 'Pantry', 55.25), ('SKU-4006', 'Outdoor', 189.00),
  ('SKU-4007', 'Kitchen', 95.00), ('SKU-4008', 'Outdoor', 121.00), ('SKU-4009', 'Pantry', 25.00),
  ('SKU-4010', 'Garden', 89.90), ('SKU-4011', 'Pantry', 59.00), ('SKU-4012', 'Kitchen', 37.00),
  ('SKU-4013', 'Outdoor', 210.00)   -- SKU-4014 has no product row
AS products(sku, category, list_price)""",
    "events": """CREATE OR REPLACE TEMP VIEW events AS SELECT * FROM VALUES
  (1, 'view', TIMESTAMP'2026-01-06 09:00:00'), (1, 'add_to_cart', TIMESTAMP'2026-01-06 09:02:00'),
  (1, 'purchase', TIMESTAMP'2026-01-06 09:10:00'), (2, 'view', TIMESTAMP'2026-01-07 10:00:00'),
  (2, 'purchase', TIMESTAMP'2026-01-07 10:40:00'), (3, 'view', TIMESTAMP'2026-01-06 11:15:00'),
  (3, 'view', TIMESTAMP'2026-01-08 11:20:00'), (4, 'add_to_cart', TIMESTAMP'2026-01-09 20:05:00'),
  (4, 'purchase', TIMESTAMP'2026-01-10 08:30:00'), (5, 'view', TIMESTAMP'2026-01-11 14:45:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00'), (5, 'purchase', TIMESTAMP'2026-01-12 12:30:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00')   -- an SDK retry committed this event twice
AS events(user_id, event, ts)""",
    "employees": """CREATE OR REPLACE TEMP VIEW employees AS SELECT * FROM VALUES
  (1,  'Ada',    'Engineering', CAST(NULL AS INT), 185000, DATE'2018-03-01'),
  (2,  'Brett',  'Engineering', 1,    150000, DATE'2019-07-15'),
  (3,  'Chen',   'Engineering', 1,    150000, DATE'2020-01-06'),
  (4,  'Dana',   'Engineering', 2,    155000, DATE'2022-09-12'),   -- out-earns her manager
  (5,  'Eli',    'Sales',       1,    130000, DATE'2019-02-01'),
  (6,  'Farah',  'Sales',       5,    95000,  DATE'2021-05-17'),
  (7,  'Gus',    'Sales',       5,    95000,  DATE'2023-01-09'),
  (8,  'Hana',   'Finance',     1,    140000, DATE'2017-11-20'),
  (9,  'Ivan',   'Finance',     8,    88000,  DATE'2024-04-01'),
  (10, 'Jo',     NULL,          8,    70000,  DATE'2025-06-30')   -- department not assigned yet
AS employees(emp_id, name, dept, manager_id, salary, hire_date)""",
}
for statement in TABLES.values():
    spark.sql(statement)

import contextlib
import io
import logging
import re


def show(query, n=50, **args):
    """Run a query and print up to n rows without truncating values. Keyword arguments bind :name parameters."""
    spark.sql(query, args=args or None).show(n, truncate=False)


def plan(query, mode="formatted"):
    """The plan EXPLAIN prints for a query, as text."""
    return spark.sql(f"EXPLAIN {mode.upper()} {query}").first()[0]


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        lines = [line for line in str(error).splitlines() if line.strip()]
        message = next((line for line in lines if re.search(r"\[[A-Z][A-Z0-9_.]+\]", line)), lines[0] if lines else "")
        print(type(error).__name__ + ":", message.strip().split(". ")[0][:120])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

## Exercises

7 easy, 6 medium, 1 hard. Write your query in the empty cell under each exercise, run it with `show(q)` or `spark.sql(q).show()`, and compare it with the expected output (columns, values and row order).

| # | exercise | difficulty | practises |
| --- | --- | --- | --- |
| 1 | [Orders by weekday](#exercise-1-orders-by-weekday) | Easy | [Extracting parts and truncating](sql-reference-08-dates-and-time-series.ipynb#extracting-parts-and-truncating), [`to_date`, `to_timestamp` and `date_format`](sql-reference-08-dates-and-time-series.ipynb#to_date-to_timestamp-and-date_format) |
| 2 | [Parse day-first dates](#exercise-2-parse-day-first-dates) | Easy | [`to_date`, `to_timestamp` and `date_format`](sql-reference-08-dates-and-time-series.ipynb#to_date-to_timestamp-and-date_format) |
| 3 | [Days from signup to first order](#exercise-3-days-from-signup-to-first-order) | Easy | [Adding, subtracting and differences](sql-reference-08-dates-and-time-series.ipynb#adding-subtracting-and-differences) |
| 4 | [Orders per hour of day](#exercise-4-orders-per-hour-of-day) | Easy | [Extracting parts and truncating](sql-reference-08-dates-and-time-series.ipynb#extracting-parts-and-truncating) |
| 5 | [Weekly revenue](#exercise-5-weekly-revenue) | Easy | [Extracting parts and truncating](sql-reference-08-dates-and-time-series.ipynb#extracting-parts-and-truncating) |
| 6 | [Orders by New York calendar day](#exercise-6-orders-by-new-york-calendar-day) | Medium | [Converting between zones](sql-reference-08-dates-and-time-series.ipynb#converting-between-zones) |
| 7 | [Daily revenue with zero days](#exercise-7-daily-revenue-with-zero-days) | Medium | [Calendar spines and gap filling](sql-reference-08-dates-and-time-series.ipynb#calendar-spines-and-gap-filling) |
| 8 | [Trailing seven-day revenue](#exercise-8-trailing-seven-day-revenue) | Medium | [Calendar spines and gap filling](sql-reference-08-dates-and-time-series.ipynb#calendar-spines-and-gap-filling) |
| 9 | [Subscription renewal dates](#exercise-9-subscription-renewal-dates) | Easy | [Adding, subtracting and differences](sql-reference-08-dates-and-time-series.ipynb#adding-subtracting-and-differences) |
| 10 | [Sessions per user](#exercise-10-sessions-per-user) | Medium | [Sessions](sql-reference-08-dates-and-time-series.ipynb#sessions) |
| 11 | [Time from first view to purchase](#exercise-11-time-from-first-view-to-purchase) | Medium | [Adding, subtracting and differences](sql-reference-08-dates-and-time-series.ipynb#adding-subtracting-and-differences) |
| 12 | [Events in 15-minute windows](#exercise-12-events-in-15-minute-windows) | Easy | [Bucketing and fixed windows](sql-reference-08-dates-and-time-series.ipynb#bucketing-and-fixed-windows) |
| 13 | [Orders outside business hours](#exercise-13-orders-outside-business-hours) | Medium | [Converting between zones](sql-reference-08-dates-and-time-series.ipynb#converting-between-zones), [Extracting parts and truncating](sql-reference-08-dates-and-time-series.ipynb#extracting-parts-and-truncating) |
| 14 | [Retention of daily cohorts](#exercise-14-retention-of-daily-cohorts) | Hard | [Cohorts and retention](sql-reference-08-dates-and-time-series.ipynb#cohorts-and-retention) |

## Exercise 1: Orders by weekday

**Difficulty:** Easy · about 5 min · **Practises:** [Extracting parts and truncating](sql-reference-08-dates-and-time-series.ipynb#extracting-parts-and-truncating), [`to_date`, `to_timestamp` and `date_format`](sql-reference-08-dates-and-time-series.ipynb#to_date-to_timestamp-and-date_format)

*Weekly seasonality starts with a weekday breakdown.*

Return `weekday` (full name, e.g. `Monday`), `orders` and `revenue`, ordered Monday first.

**Expected output**

| weekday | orders | revenue |
| --- | --- | --- |
| Monday | 5 | 1488.50 |
| Tuesday | 2 | 470.00 |
| Wednesday | 3 | 300.65 |
| Friday | 1 | 310.00 |
| Saturday | 1 | 89.90 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT date_format(order_ts, 'EEEE') AS weekday, count(*) AS orders, sum(amount) AS revenue
    FROM orders
    GROUP BY date_format(order_ts, 'EEEE'), extract(DOW_ISO FROM order_ts)
    ORDER BY extract(DOW_ISO FROM order_ts)
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 2: Parse day-first dates

**Difficulty:** Easy · about 5 min · **Practises:** [`to_date`, `to_timestamp` and `date_format`](sql-reference-08-dates-and-time-series.ipynb#to_date-to_timestamp-and-date_format)

*European exports write the day first; a wrong pattern swaps day and month silently.*

Using `uploads` below, return `file` and `upload_date` parsed from `dd/MM/yyyy` (NULL when invalid), ordered by `upload_date` with invalid rows last, then `file`.

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW uploads AS SELECT * FROM VALUES
  ('a.csv', '05/01/2026'), ('b.csv', '31/12/2025'), ('c.csv', '2026-01-05'), ('d.csv', '01/02/2026')
AS uploads(file, day_text)
""")

**Expected output**

| file | upload_date |
| --- | --- |
| b.csv | 2025-12-31 |
| a.csv | 2026-01-05 |
| d.csv | 2026-02-01 |
| c.csv | null |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT file, CAST(try_to_timestamp(day_text, 'dd/MM/yyyy') AS DATE) AS upload_date
    FROM uploads
    ORDER BY upload_date NULLS LAST, file
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 3: Days from signup to first order

**Difficulty:** Easy · about 5 min · **Practises:** [Adding, subtracting and differences](sql-reference-08-dates-and-time-series.ipynb#adding-subtracting-and-differences)

*Time-to-first-purchase is a core activation metric.*

For customers who ordered, return `customer_id`, `signup_date`, `first_order_date` and `days_to_first_order`, ordered by `days_to_first_order`, then `customer_id`.

**Expected output**

| customer_id | signup_date | first_order_date | days_to_first_order |
| --- | --- | --- | --- |
| 6 | 2024-08-01 | 2026-01-13 | 530 |
| 5 | 2024-06-20 | 2026-01-12 | 571 |
| 4 | 2024-05-02 | 2026-01-10 | 618 |
| 3 | 2024-03-15 | 2026-01-07 | 663 |
| 2 | 2024-02-11 | 2026-01-06 | 695 |
| 1 | 2024-01-03 | 2026-01-05 | 733 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT c.customer_id, c.signup_date, to_date(min(o.order_ts)) AS first_order_date,
           datediff(to_date(min(o.order_ts)), c.signup_date) AS days_to_first_order
    FROM customers c
    JOIN orders o ON o.customer_id = c.customer_id
    GROUP BY c.customer_id, c.signup_date
    ORDER BY days_to_first_order, c.customer_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 4: Orders per hour of day

**Difficulty:** Easy · about 3 min · **Practises:** [Extracting parts and truncating](sql-reference-08-dates-and-time-series.ipynb#extracting-parts-and-truncating)

*Staffing and capacity planning by hour.*

Return `hour` (0-23 in UTC) and `orders` for hours with at least one order, ordered by `hour`.

**Expected output**

| hour | orders |
| --- | --- |
| 0 | 1 |
| 8 | 2 |
| 9 | 1 |
| 10 | 2 |
| 11 | 1 |
| 12 | 1 |
| 13 | 1 |
| 15 | 1 |
| 17 | 1 |
| 19 | 1 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT hour(order_ts) AS hour, count(*) AS orders
    FROM orders
    GROUP BY hour(order_ts)
    ORDER BY hour
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 5: Weekly revenue

**Difficulty:** Easy · about 5 min · **Practises:** [Extracting parts and truncating](sql-reference-08-dates-and-time-series.ipynb#extracting-parts-and-truncating)

*Weeks start on Monday in ISO reporting.*

Return `week_start` (the Monday, as a date) and `revenue` per week, ordered by `week_start`.

**Expected output**

| week_start | revenue |
| --- | --- |
| 2026-01-05 | 1098.65 |
| 2026-01-12 | 1560.40 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT to_date(date_trunc('week', order_ts)) AS week_start, sum(amount) AS revenue
    FROM orders
    GROUP BY to_date(date_trunc('week', order_ts))
    ORDER BY week_start
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 6: Orders by New York calendar day

**Difficulty:** Medium · about 10 min · **Practises:** [Converting between zones](sql-reference-08-dates-and-time-series.ipynb#converting-between-zones)

*A US business reports by its local day, not by UTC.*

Return `ny_date` (the order's date in `America/New_York`) and `orders`, ordered by `ny_date`.

**Expected output**

| ny_date | orders |
| --- | --- |
| 2026-01-05 | 3 |
| 2026-01-06 | 1 |
| 2026-01-07 | 2 |
| 2026-01-09 | 1 |
| 2026-01-10 | 1 |
| 2026-01-12 | 2 |
| 2026-01-13 | 2 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT to_date(from_utc_timestamp(order_ts, 'America/New_York')) AS ny_date, count(*) AS orders
    FROM orders
    GROUP BY to_date(from_utc_timestamp(order_ts, 'America/New_York'))
    ORDER BY ny_date
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 7: Daily revenue with zero days

**Difficulty:** Medium · about 10 min · **Practises:** [Calendar spines and gap filling](sql-reference-08-dates-and-time-series.ipynb#calendar-spines-and-gap-filling)

*Charts and averages are wrong when empty days are missing.*

Return `day`, `orders` and `revenue` for every day from 2026-01-05 to 2026-01-14 inclusive, with 0 on days without orders, ordered by `day`.

**Expected output**

| day | orders | revenue |
| --- | --- | --- |
| 2026-01-05 | 3 | 234.50 |
| 2026-01-06 | 1 | 260.00 |
| 2026-01-07 | 2 | 204.25 |
| 2026-01-08 | 0 | 0.00 |
| 2026-01-09 | 1 | 310.00 |
| 2026-01-10 | 1 | 89.90 |
| 2026-01-11 | 0 | 0.00 |
| 2026-01-12 | 2 | 1254.00 |
| 2026-01-13 | 1 | 210.00 |
| 2026-01-14 | 1 | 96.40 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    WITH spine AS (
        SELECT explode(sequence(DATE'2026-01-05', DATE'2026-01-14', INTERVAL 1 DAY)) AS day
    ),
    daily AS (
        SELECT to_date(order_ts) AS day, count(*) AS orders, sum(amount) AS revenue
        FROM orders GROUP BY to_date(order_ts)
    )
    SELECT s.day, coalesce(d.orders, 0) AS orders, coalesce(d.revenue, 0) AS revenue
    FROM spine s
    LEFT JOIN daily d ON d.day = s.day
    ORDER BY s.day
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 8: Trailing seven-day revenue

**Difficulty:** Medium · about 10 min · **Practises:** [Calendar spines and gap filling](sql-reference-08-dates-and-time-series.ipynb#calendar-spines-and-gap-filling)

*Rolling windows must count calendar days, not rows.*

Per day with orders, return `day`, `revenue` and `trailing_7d`: revenue over that day and the six days before it (a calendar window), ordered by `day`.

**Expected output**

| day | revenue | trailing_7d |
| --- | --- | --- |
| 2026-01-05 | 234.50 | 234.50 |
| 2026-01-06 | 260.00 | 494.50 |
| 2026-01-07 | 204.25 | 698.75 |
| 2026-01-09 | 310.00 | 1008.75 |
| 2026-01-10 | 89.90 | 1098.65 |
| 2026-01-12 | 1254.00 | 2118.15 |
| 2026-01-13 | 210.00 | 2068.15 |
| 2026-01-14 | 96.40 | 1960.30 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    WITH daily AS (
        SELECT to_date(order_ts) AS day, sum(amount) AS revenue
        FROM orders GROUP BY to_date(order_ts)
    )
    SELECT day, revenue,
           sum(revenue) OVER (ORDER BY day RANGE BETWEEN INTERVAL 6 DAYS PRECEDING AND CURRENT ROW) AS trailing_7d
    FROM daily
    ORDER BY day
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 9: Subscription renewal dates

**Difficulty:** Easy · about 5 min · **Practises:** [Adding, subtracting and differences](sql-reference-08-dates-and-time-series.ipynb#adding-subtracting-and-differences)

*Monthly billing must handle short months.*

Using `subscriptions` below, return `sub_id`, `started` and the next three monthly renewal dates `renewal_1`, `renewal_2` and `renewal_3` (add 1, 2 and 3 months to `started`), ordered by `sub_id`.

In [ ]:
spark.sql("CREATE OR REPLACE TEMP VIEW subscriptions AS SELECT * FROM VALUES (1, DATE'2026-01-31'), (2, DATE'2026-02-15'), (3, DATE'2027-11-30') AS s(sub_id, started)")

**Expected output**

| sub_id | started | renewal_1 | renewal_2 | renewal_3 |
| --- | --- | --- | --- | --- |
| 1 | 2026-01-31 | 2026-02-28 | 2026-03-31 | 2026-04-30 |
| 2 | 2026-02-15 | 2026-03-15 | 2026-04-15 | 2026-05-15 |
| 3 | 2027-11-30 | 2027-12-30 | 2028-01-30 | 2028-02-29 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT sub_id, started,
           add_months(started, 1) AS renewal_1,
           add_months(started, 2) AS renewal_2,
           add_months(started, 3) AS renewal_3
    FROM subscriptions
    ORDER BY sub_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 10: Sessions per user

**Difficulty:** Medium · about 15 min · **Practises:** [Sessions](sql-reference-08-dates-and-time-series.ipynb#sessions)

*Sessionisation with a 30-minute inactivity gap.*

Return `user_id`, `sessions` and `events` (distinct events, ignoring exact duplicates) where a new session starts when more than 30 minutes pass since the user's previous event. Order by `user_id`.

**Expected output**

| user_id | sessions | events |
| --- | --- | --- |
| 1 | 1 | 3 |
| 2 | 2 | 2 |
| 3 | 2 | 2 |
| 4 | 2 | 2 |
| 5 | 3 | 3 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    WITH dedup AS (SELECT DISTINCT user_id, event, ts FROM events),
    flagged AS (
        SELECT user_id, ts,
               CASE WHEN lag(ts) OVER (PARTITION BY user_id ORDER BY ts) IS NULL
                      OR ts > lag(ts) OVER (PARTITION BY user_id ORDER BY ts) + INTERVAL 30 MINUTES
                    THEN 1 ELSE 0 END AS new_session
        FROM dedup
    )
    SELECT user_id, sum(new_session) AS sessions, count(*) AS events
    FROM flagged
    GROUP BY user_id
    ORDER BY user_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 11: Time from first view to purchase

**Difficulty:** Medium · about 10 min · **Practises:** [Adding, subtracting and differences](sql-reference-08-dates-and-time-series.ipynb#adding-subtracting-and-differences)

*Funnel latency per user.*

For users with both a view and a purchase, return `user_id`, `first_view`, `first_purchase` and `minutes_to_purchase` (whole minutes), ordered by `user_id`.

**Expected output**

| user_id | first_view | first_purchase | minutes_to_purchase |
| --- | --- | --- | --- |
| 1 | 2026-01-06 09:00:00 | 2026-01-06 09:10:00 | 10 |
| 2 | 2026-01-07 10:00:00 | 2026-01-07 10:40:00 | 40 |
| 5 | 2026-01-11 14:45:00 | 2026-01-12 12:30:00 | 1305 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT user_id,
           min(ts) FILTER (WHERE event = 'view') AS first_view,
           min(ts) FILTER (WHERE event = 'purchase') AS first_purchase,
           timestampdiff(MINUTE, min(ts) FILTER (WHERE event = 'view'), min(ts) FILTER (WHERE event = 'purchase')) AS minutes_to_purchase
    FROM events
    GROUP BY user_id
    HAVING count_if(event = 'view') > 0 AND count_if(event = 'purchase') > 0
    ORDER BY user_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 12: Events in 15-minute windows

**Difficulty:** Easy · about 5 min · **Practises:** [Bucketing and fixed windows](sql-reference-08-dates-and-time-series.ipynb#bucketing-and-fixed-windows)

*Fixed windows are the batch form of streaming windows.*

For 2026-01-06 and 2026-01-12, return `window_start`, `window_end` and `events` per 15-minute window that has events (count every row), ordered by `window_start`.

**Expected output**

| window_start | window_end | events |
| --- | --- | --- |
| 2026-01-06 09:00:00 | 2026-01-06 09:15:00 | 3 |
| 2026-01-06 11:15:00 | 2026-01-06 11:30:00 | 1 |
| 2026-01-12 09:00:00 | 2026-01-12 09:15:00 | 2 |
| 2026-01-12 12:30:00 | 2026-01-12 12:45:00 | 1 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT window(ts, '15 minutes').start AS window_start,
           window(ts, '15 minutes').end   AS window_end,
           count(*) AS events
    FROM events
    WHERE to_date(ts) IN (DATE'2026-01-06', DATE'2026-01-12')
    GROUP BY window(ts, '15 minutes')
    ORDER BY window_start
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 13: Orders outside business hours

**Difficulty:** Medium · about 10 min · **Practises:** [Converting between zones](sql-reference-08-dates-and-time-series.ipynb#converting-between-zones), [Extracting parts and truncating](sql-reference-08-dates-and-time-series.ipynb#extracting-parts-and-truncating)

*Support coverage depends on local business hours.*

Return `order_id`, `local_ts` (New York time) and `reason` for orders placed outside Monday–Friday 09:00–17:00 New York time: `weekend` or `after hours`. Order by `order_id`.

**Expected output**

| order_id | local_ts | reason |
| --- | --- | --- |
| 1 | 2026-01-05 04:15:00 | after hours |
| 2 | 2026-01-05 05:05:00 | after hours |
| 3 | 2026-01-05 06:40:00 | after hours |
| 5 | 2026-01-07 03:05:00 | after hours |
| 7 | 2026-01-09 07:10:00 | after hours |
| 8 | 2026-01-10 14:30:00 | weekend |
| 9 | 2026-01-12 03:55:00 | after hours |
| 10 | 2026-01-12 08:00:00 | after hours |
| 11 | 2026-01-13 05:25:00 | after hours |
| 12 | 2026-01-13 19:30:00 | after hours |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    WITH local AS (
        SELECT order_id, from_utc_timestamp(order_ts, 'America/New_York') AS local_ts FROM orders
    )
    SELECT order_id, local_ts,
           CASE WHEN extract(DOW_ISO FROM local_ts) >= 6 THEN 'weekend' ELSE 'after hours' END AS reason
    FROM local
    WHERE extract(DOW_ISO FROM local_ts) >= 6 OR hour(local_ts) < 9 OR hour(local_ts) >= 17
    ORDER BY order_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 14: Retention of daily cohorts

**Difficulty:** Hard · about 20 min · **Practises:** [Cohorts and retention](sql-reference-08-dates-and-time-series.ipynb#cohorts-and-retention)

*Cohort retention tables are a staple product-analytics interview question.*

Group users by the day of their first event (`cohort_day`). Return `cohort_day`, `users`, `day_1` (users active on exactly the next day) and `retained_pct` = 100 × day_1 / users rounded to 1, ordered by `cohort_day`.

**Expected output**

| cohort_day | users | day_1 | retained_pct |
| --- | --- | --- | --- |
| 2026-01-06 | 2 | 0 | 0.0 |
| 2026-01-07 | 1 | 0 | 0.0 |
| 2026-01-09 | 1 | 1 | 100.0 |
| 2026-01-11 | 1 | 1 | 100.0 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    WITH activity AS (SELECT DISTINCT user_id, to_date(ts) AS day FROM events),
    cohorts AS (SELECT user_id, min(day) AS cohort_day FROM activity GROUP BY user_id),
    joined AS (
        SELECT c.cohort_day, a.user_id, datediff(a.day, c.cohort_day) AS day_n
        FROM activity a JOIN cohorts c ON c.user_id = a.user_id
    )
    SELECT cohort_day,
           count(DISTINCT user_id) AS users,
           count(DISTINCT CASE WHEN day_n = 1 THEN user_id END) AS day_1,
           round(100 * count(DISTINCT CASE WHEN day_n = 1 THEN user_id END) / count(DISTINCT user_id), 1) AS retained_pct
    FROM joined
    GROUP BY cohort_day
    ORDER BY cohort_day
"""
result = spark.sql(q)
result.show(truncate=False)

**Next:** look anything up in [SQL reference 8: dates, times and time series](sql-reference-08-dates-and-time-series.ipynb), or try mixed interview problems in [SQL interview drills](sql-13-interview-drills.ipynb).